# Hugging Face Applications — Lesson 2: Summarization

> Learning material for **Hugging Face Applications**. Companion to the lesson script `02_Summarization.py` (same content, runnable without Jupyter).

**Task ID:** HF-202  |  **Folder:** `documentation`


## What is summarization?

A **summarization model** reads a long text and produces a short text that keeps the important meaning.

> 📖 `"Hugging Face ... the community has uploaded more than a million artifacts"`
>
> ➜ 💬 `"Hugging Face is an open-source AI company whose Hub hosts over a million models."`

## The idea: encoder → decoder (seq2seq)

BART-style models are **sequence-to-sequence (seq2seq)** — the word *seq2seq*
means they map one sequence of words to another sequence of words:

1. **Encoder** reads the *whole* input in one go and builds a compressed understanding of it (the "context vector").
2. **Decoder** writes the summary word by word, looking at that understanding (this is the same next-word loop as in lesson HF-201).

This is different from causal models (GPT), which only ever look at the text *before* the next word.

> **Analogy:** encoder = you read the whole article; decoder = you retell it in one or two sentences.

## transformers v5 note

In transformers 5.x the `pipeline("summarization")` shortcut was removed. The **tokenize → generate → decode** loop below is the supported way, and it shows everything the old pipeline hid.

**Step 1 — load tokenizer + model:**


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# distilbart: a compressed (distilled) BART, fast on CPU.
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Loaded", model_name)


**Step 2 — tokenize.** The tokenizer turns words into number IDs (the only thing the model understands) and back:


In [ ]:
sentence = "The quick brown fox jumps over the lazy dog."
ids = tokenizer(sentence)["input_ids"]
print("IDs:", ids)
print("Back to words:", tokenizer.decode(ids))


**Step 3 — the three-step loop.** Tokenize → `model.generate()` → decode.


In [ ]:
long_text = (
    "Hugging Face is a company based in New York that builds open-source tools "
    "for machine learning. Its Transformers library lets developers use thousands "
    "of pretrained models with a few lines of code. The library supports text, "
    "vision and audio tasks, and it runs on PyTorch, TensorFlow and JAX. Models "
    "are shared on the Hugging Face Hub, where the community has uploaded more "
    "than a million artifacts."
)

inputs = tokenizer(long_text, return_tensors="pt", truncation=True, max_length=1024)
output_ids = model.generate(
    inputs["input_ids"],
    min_length=25,   # summary must be at least this long (tokens)
    max_length=70,   # and at most this long
    num_beams=2,     # beam search width (see below)
    early_stopping=True,
)
summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Original:", long_text)
print()
print("Summary:", summary)


## The knobs

| Parameter | What it does | 
|-----------|--------------|
| `truncation=True, max_length=1024` | clip input longer than BART's context window (~1024 tokens) | 
| `min_length` / `max_length` | the summary size, in tokens | 
| `num_beams=2` | **beam search**: track the 2 best partial texts instead of 1. More beams = better but slower. | 
| `early_stopping=True` | stop as soon as the beams agree on a finished summary |

Try `num_beams=4` — notice the quality/quality-balance shift for longer summaries.

## What if my document is longer than 1024 tokens?

Chunk it: cut the text into pieces, summarize each piece, concatenate the piece summaries. (Real products also re-summarize the summaries.)


In [ ]:
def chunk_summarize(text, chunk_size=900):
    words = text.split()
    pieces = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    summaries = []
    for piece in pieces:
        ids = tokenizer(piece, return_tensors="pt", truncation=True, max_length=1024)
        out = model.generate(ids["input_ids"], max_length=60, min_length=20, num_beams=2,
                             early_stopping=True)
        summaries.append(tokenizer.decode(out[0], skip_special_tokens=True))
    return " ".join(summaries)

long_article = " ".join([long_text] * 5)   # pretend it's very long
print(chunk_summarize(long_article))


## Try it yourself

1. Summarize a Wikipedia article you like (paste it into `long_text`).
2. Shrink the summary to `min_length=10, max_length=25` — what happens?
3. Swap the model for `facebook/bart-large-cnn` (better, bigger download).

## Common pitfalls

- **Model ignores min_length on very short input** — normal; the decoder can't pad with meaning.
- **Summary repeats itself** — lower `max_length` or raise beam count.
- **Long inputs are silently cut** — always set `truncation=True` and know the model's limit.

## Summary

- Seq2seq = encoder reads everything, decoder writes the new text.
- The loop is always: tokenize → `generate()` → decode.
- Length and beam size are the two dials that matter.

**Next lesson:** HF-203 — Question Answering.  |  Extra reading: `../resources/reference_links.md`
